In [1]:
library(fixest)
library(dplyr)
library(tidyr)
library(ggplot2)
library(lubridate)
library(modelsummary)

# ---- 1. Load panel --------------------------------------------------------
panel <- read.csv("../data/monthly_panel_mttu_mttr.csv")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




## **MTTU Code**

In [ ]:
# ============================================================================
# PPML TWFE — Linear Model for Aggregate Score Only
# ============================================================================

# ---- Run Model ----
model_aggregate <- feglm( MTTU ~ Aggregate_score +
  log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) +
  version_age_days + log1p(Release_count) |
  package_name + year_month, data = panel,
                         family = "poisson", cluster = ~package_name)


summary(model_aggregate)

# ---- Calculate and Print % Change ----
# Calculate the exact percentage change for a 1-unit increase
pct_change <- (exp(coef(model_aggregate)["score"]) - 1) * 100

cat("\n------------------------------------------------------------\n")
cat(sprintf("A 1-unit increase in Aggregate_score is associated with a %.2f%% change in MTTU.\n", 
            pct_change))
cat("------------------------------------------------------------\n")

NOTES: 1,837 observations removed because of NA values (RHS: 1,837).
       5,272/0 fixed-effects (15,926 observations) removed because of only 0 outcomes or singletons.



GLM estimation, family = poisson, Dep. Var.: MTTU
Observations: 61,300
Fixed-effects: package_name: 8,275,  year_month: 25
Standard-errors: Clustered (package_name) 
                              Estimate Std. Error   z value   Pr(>|z|)    
Aggregate_score              -0.155508   0.015955  -9.74687  < 2.2e-16 ***
log1p(downloads_7_day_total) -0.006983   0.003523  -1.98205 4.7474e-02 *  
log1p(dependents)             0.023095   0.009851   2.34431 1.9062e-02 *  
log1p(loc)                   -0.070276   0.036282  -1.93695 5.2751e-02 .  
version_age_days              0.003685   0.000583   6.31582 2.6874e-10 ***
log1p(Release_count)         -0.139270   0.013634 -10.21514  < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -152,458.1   Adj. Pseudo R2: 0.706144
           BIC:  396,466.6     Squared Cor.: 0.858595


------------------------------------------------------------
A 1-unit increase in Aggregate_score is associated with a NA% change in MTTU.
------------------------------------------------------------


## **Lag: 1 Month, 3 Months, 6 Months**

In [2]:
library(fixest)
library(dplyr)
library(tidyr)
library(ggplot2)
library(lubridate)
library(modelsummary)

# ---- 1. Load panel --------------------------------------------------------
panel <- read.csv("../data/monthly_panel_mttu_mttr.csv")

# ---- 2. Create lagged Aggregate_score variables ---------------------------
# Assumes year_month can be parsed as a date (e.g., "2021-01" or "2021-01-01")
# and that the panel has one row per package_name x year_month.
panel <- panel %>%
  mutate(year_month_date = ym(substr(year_month, 1, 7))) %>%  # adjust if format differs
  arrange(package_name, year_month_date) %>%
  group_by(package_name) %>%
  mutate(
    Aggregate_score_lag1 = dplyr::lag(Aggregate_score, n = 1, order_by = year_month_date),
    Aggregate_score_lag3 = dplyr::lag(Aggregate_score, n = 3, order_by = year_month_date),
    Aggregate_score_lag6 = dplyr::lag(Aggregate_score, n = 6, order_by = year_month_date)
  ) %>%
  ungroup()

# ============================================================================
# PPML TWFE — Lagged Aggregate Score Models (1, 3, 6 months)
# ============================================================================

model_lag1 <- feglm(
  MTTU ~ Aggregate_score_lag1 +
    log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) +
    version_age_days + log1p(Release_count) |
    package_name + year_month,
  data = panel, family = "poisson", cluster = ~package_name
)

model_lag3 <- feglm(
  MTTU ~ Aggregate_score_lag3 +
    log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) +
    version_age_days + log1p(Release_count) |
    package_name + year_month,
  data = panel, family = "poisson", cluster = ~package_name
)

model_lag6 <- feglm(
  MTTU ~ Aggregate_score_lag6 +
    log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) +
    version_age_days + log1p(Release_count) |
    package_name + year_month,
  data = panel, family = "poisson", cluster = ~package_name
)

# ---- 3. Combined summary table ---------------------------------------
lag_models <- list(
  "1-month lag" = model_lag1,
  "3-month lag" = model_lag3,
  "6-month lag" = model_lag6
)

modelsummary(lag_models, stars = TRUE)

# ---- 4. % change for each lag ------------------------------------------
for (name in names(lag_models)) {
  m <- lag_models[[name]]
  coef_name <- names(coef(m))[1]  # the Aggregate_score_lagX term
  pct_change <- (exp(coef(m)[coef_name]) - 1) * 100
  cat("\n------------------------------------------------------------\n")
  cat(sprintf("%s: A 1-unit increase in Aggregate Score (%s) is associated with a %.2f%% change in MTTU.\n",
              name, coef_name, pct_change))
  cat("------------------------------------------------------------\n")
}

NOTES: 15,212 observations removed because of NA values (RHS: 15,212).
       3,929/0 fixed-effects (12,480 observations) removed because of only 0 outcomes or singletons.

NOTES: 33,979 observations removed because of NA values (RHS: 33,979).
       2,174/0 fixed-effects (7,910 observations) removed because of only 0 outcomes or singletons.

NOTES: 51,234 observations removed because of NA values (RHS: 51,234).
       1,165/0 fixed-effects (4,475 observations) removed because of only 0 outcomes or singletons.




+------------------------------+-------------+-------------+-------------+
|                              | 1-month lag | 3-month lag | 6-month lag |
+==============================+=============+=============+=============+
| Aggregate_score_lag1         | -0.112***   |             |             |
+------------------------------+-------------+-------------+-------------+
|                              | (0.012)     |             |             |
+------------------------------+-------------+-------------+-------------+
| log1p(downloads_7_day_total) | -0.006+     | -0.006+     | -0.011**    |
+------------------------------+-------------+-------------+-------------+
|                              | (0.003)     | (0.003)     | (0.004)     |
+------------------------------+-------------+-------------+-------------+
| log1p(dependents)            | 0.010       | 0.013+      | 0.009       |
+------------------------------+-------------+-------------+-------------+
|                       


------------------------------------------------------------
1-month lag: A 1-unit increase in Aggregate Score (Aggregate_score_lag1) is associated with a -10.60% change in MTTU.
------------------------------------------------------------

------------------------------------------------------------
3-month lag: A 1-unit increase in Aggregate Score (Aggregate_score_lag3) is associated with a -7.06% change in MTTU.
------------------------------------------------------------

------------------------------------------------------------
6-month lag: A 1-unit increase in Aggregate Score (Aggregate_score_lag6) is associated with a -4.02% change in MTTU.
------------------------------------------------------------


In [3]:
glimpse(panel)

Rows: 79,063
Columns: 25
$ package_name              <chr> "0g", "0g", "0http", "0http", "0http", "0htt…
$ year_month                <chr> "2024-04", "2024-05", "2024-04", "2025-01", …
$ MTTU                      <dbl> 0.000000, 0.000000, 94.135218, 132.616153, 1…
$ MTTR                      <dbl> 0.000000, 0.000000, 0.000000, 0.000000, 0.00…
$ Release_count             <int> 2, 11, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 3…
$ Aggregate_score           <dbl> 3.450000, 4.000000, 3.000000, 3.200000, 3.60…
$ Binary_Artifacts_score    <dbl> 5.5, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.…
$ Code_Review_score         <dbl> 0.000000, 0.000000, 0.000000, 0.000000, 0.00…
$ Contrib_score             <dbl> 3, 3, 0, 0, 0, 0, 10, 3, 3, 3, 3, 0, 0, 0, 0…
$ Dangerous_Workflow_score  <dbl> 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, …
$ DependUpTool_score        <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 10, 0, 0, 0, 0…
$ Fuzzing_score             <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…
$ License_score

## Making the panel continuous per package

In [4]:
panel <- panel %>%
  mutate(year_month_date = ym(substr(year_month, 1, 7))) %>%
  arrange(package_name, year_month_date)

# Build a complete monthly grid per package so gaps are explicit
panel_complete <- panel %>%
  group_by(package_name) %>%
  complete(year_month_date = seq(min(year_month_date), max(year_month_date), by = "month")) %>%
  ungroup() %>%
  arrange(package_name, year_month_date)

# Now lags reflect true calendar distance
panel_complete <- panel_complete %>%
  group_by(package_name) %>%
  mutate(
    Aggregate_score_lag1 = dplyr::lag(Aggregate_score, n = 1),
    Aggregate_score_lag3 = dplyr::lag(Aggregate_score, n = 3),
    Aggregate_score_lag6 = dplyr::lag(Aggregate_score, n = 6)
  ) %>%
  ungroup()

# Drop the synthetic gap-filler rows before modeling —
# keep only rows that existed in your original panel
panel_final <- panel_complete %>%
  semi_join(panel, by = c("package_name", "year_month_date"))

In [5]:
# ---- Sanity check: confirm gaps are now explicit and lags respect them ----

# 1. Pick a package you know has a gap (e.g., "0http")
check_pkg <- panel_complete %>%
  filter(package_name == "0http") %>%
  arrange(year_month_date) %>%
  select(package_name, year_month_date, Aggregate_score,
         Aggregate_score_lag1, Aggregate_score_lag3, Aggregate_score_lag6)

print(check_pkg, n = Inf)

# 2. Confirm the grid is truly continuous per package (no skipped months)
gap_check <- panel_complete %>%
  group_by(package_name) %>%
  summarise(
    n_rows = n(),
    expected_rows = as.integer(interval(min(year_month_date), max(year_month_date)) %/% months(1)) + 1
  ) %>%
  filter(n_rows != expected_rows)

cat(sprintf("Packages with unexpected row counts after complete(): %d\n", nrow(gap_check)))

# 3. Confirm panel_final only contains rows that existed in the original panel
cat(sprintf("Original panel rows: %d\n", nrow(panel)))
cat(sprintf("panel_final rows (should match original): %d\n", nrow(panel_final)))

# 4. Spot-check: for a real (non-gap) row with a known prior gap,
# confirm lag1 is NA if the immediately preceding calendar month had no release
spot_check <- panel_final %>%
  filter(package_name == "0http") %>%
  arrange(year_month_date) %>%
  select(year_month_date, Aggregate_score, Aggregate_score_lag1)

print(spot_check, n = Inf)

# A tibble: 14 × 6
   package_name year_month_date Aggregate_score Aggregate_score_lag1
   <chr>        <date>                    <dbl>                <dbl>
 1 0http        2024-04-01                  3                   NA  
 2 0http        2024-05-01                 NA                    3  
 3 0http        2024-06-01                 NA                   NA  
 4 0http        2024-07-01                 NA                   NA  
 5 0http        2024-08-01                 NA                   NA  
 6 0http        2024-09-01                 NA                   NA  
 7 0http        2024-10-01                 NA                   NA  
 8 0http        2024-11-01                 NA                   NA  
 9 0http        2024-12-01                 NA                   NA  
10 0http        2025-01-01                  3.2                 NA  
11 0http        2025-02-01                  3.6                  3.2
12 0http        2025-03-01                 NA                    3.6
13 0http       

## Updated Pipeline

In [10]:
library(fixest)
library(dplyr)
library(tidyr)
library(ggplot2)
library(lubridate)
library(modelsummary)

# ---- 1. Load panel --------------------------------------------------------
panel <- read.csv("../data/monthly_panel_mttu_mttr.csv")

panel <- panel %>%
  mutate(year_month_date = ym(substr(year_month, 1, 7))) %>%
  arrange(package_name, year_month_date)

# ---- 2. Build a complete monthly grid per package so gaps are explicit ----
panel_complete <- panel %>%
  group_by(package_name) %>%
  complete(year_month_date = seq(min(year_month_date), max(year_month_date), by = "month")) %>%
  ungroup() %>%
  arrange(package_name, year_month_date)

# ---- 3. Lag Aggregate_score by true calendar distance ---------------------
panel_complete <- panel_complete %>%
  group_by(package_name) %>%
  mutate(
    Aggregate_score_lag1 = dplyr::lag(Aggregate_score, n = 1),
    Aggregate_score_lag3 = dplyr::lag(Aggregate_score, n = 3),
    Aggregate_score_lag6 = dplyr::lag(Aggregate_score, n = 6),
    Aggregate_score_lag12 = dplyr::lag(Aggregate_score, n = 12),
  ) %>%
  ungroup()

# ---- 4. Drop synthetic gap-filler rows — keep only real observations -----
panel_final <- panel_complete %>%
  semi_join(panel, by = c("package_name", "year_month_date"))

# ============================================================================
# PPML TWFE — Lagged Aggregate Score Models (1, 3, 6 months)
# ============================================================================

model_lag1 <- feglm(
  MTTU ~ Aggregate_score_lag1 +
    log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) +
    version_age_days + log1p(Release_count) |
    package_name + year_month,
  data = panel_final, family = "poisson", cluster = ~package_name
)

model_lag3 <- feglm(
  MTTU ~ Aggregate_score_lag3 +
    log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) +
    version_age_days + log1p(Release_count) |
    package_name + year_month,
  data = panel_final, family = "poisson", cluster = ~package_name
)

model_lag6 <- feglm(
  MTTU ~ Aggregate_score_lag6 +
    log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) +
    version_age_days + log1p(Release_count) |
    package_name + year_month,
  data = panel_final, family = "poisson", cluster = ~package_name
)

model_lag12 <- feglm(
  MTTU ~ Aggregate_score_lag12 +
    log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) +
    version_age_days + log1p(Release_count) |
    package_name + year_month,
  data = panel_final, family = "poisson", cluster = ~package_name
)

# ---- 5. Combined summary table --------------------------------------------
lag_models <- list(
  "1-month lag" = model_lag1,
  "3-month lag" = model_lag3,
  "6-month lag" = model_lag6,
  "12-month lag" = model_lag12
)

modelsummary(lag_models, stars = TRUE)

# ---- 6. % change and effective N for each lag ------------------------------
for (name in names(lag_models)) {
  m <- lag_models[[name]]
  coef_name <- names(coef(m))[1]  # the Aggregate_score_lagX term
  pct_change <- (exp(coef(m)[coef_name]) - 1) * 100
  n_obs <- nobs(m)
  cat("\n------------------------------------------------------------\n")
  cat(sprintf("%s (N = %d): A 1-unit increase in Aggregate Score (%s) is associated with a %.2f%% change in MTTU.\n",
              name, n_obs, coef_name, pct_change))
  cat("------------------------------------------------------------\n")
}

NOTES: 40,627 observations removed because of NA values (RHS: 40,627).
       3,181/0 fixed-effects (7,640 observations) removed because of only 0 outcomes or singletons.

NOTES: 45,630 observations removed because of NA values (RHS: 45,630).
       2,890/0 fixed-effects (6,692 observations) removed because of only 0 outcomes or singletons.

NOTES: 52,289 observations removed because of NA values (RHS: 52,289).
       2,611/0 fixed-effects (5,532 observations) removed because of only 0 outcomes or singletons.

NOTES: 62,770 observations removed because of NA values (RHS: 62,770).
       2,255/0 fixed-effects (3,998 observations) removed because of only 0 outcomes or singletons.




+------------------------------+-------------+-------------+-------------+--------------+
|                              | 1-month lag | 3-month lag | 6-month lag | 12-month lag |
+==============================+=============+=============+=============+==============+
| Aggregate_score_lag1         | -0.108***   |             |             |              |
+------------------------------+-------------+-------------+-------------+--------------+
|                              | (0.017)     |             |             |              |
+------------------------------+-------------+-------------+-------------+--------------+
| log1p(downloads_7_day_total) | -0.005      | 0.003       | -0.003      | -0.021**     |
+------------------------------+-------------+-------------+-------------+--------------+
|                              | (0.004)     | (0.004)     | (0.004)     | (0.006)      |
+------------------------------+-------------+-------------+-------------+--------------+
| log1p(d


------------------------------------------------------------
1-month lag (N = 30796): A 1-unit increase in Aggregate Score (Aggregate_score_lag1) is associated with a -10.26% change in MTTU.
------------------------------------------------------------

------------------------------------------------------------
3-month lag (N = 26741): A 1-unit increase in Aggregate Score (Aggregate_score_lag3) is associated with a -8.06% change in MTTU.
------------------------------------------------------------

------------------------------------------------------------
6-month lag (N = 21242): A 1-unit increase in Aggregate Score (Aggregate_score_lag6) is associated with a -3.24% change in MTTU.
------------------------------------------------------------

------------------------------------------------------------
12-month lag (N = 12295): A 1-unit increase in Aggregate Score (Aggregate_score_lag12) is associated with a -2.24% change in MTTU.
--------------------------------------------------